In [1]:
!rm .fleche -rf

# Getting Started with Fleche

This notebook demonstrates the main features of the `fleche` library, a caching library for Python.

## Long-running calculation

In [2]:
import time
from fleche import fleche, cache, tags
from fleche.digest import Digest

Using default memory cache: no config file found


In [3]:
@fleche
def long_running_calculation(x):
    print(f'Running calculation for {x}...')
    time.sleep(2)
    return x * x

In [4]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'First call took {end - start:.2f} seconds.')

Running calculation for 2...


First call took 2.00 seconds.


In [5]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'Second call took {end - start:.2f} seconds.')

Second call took 0.00 seconds.


In [6]:
start = time.time()
long_running_calculation(100)
end = time.time()
print(f'Second call took {end - start:.2f} seconds.')

Running calculation for 100...


Second call took 2.00 seconds.


As you can see, the second call returns almost instantly, because the result was cached.
As soon as the argument changes, fleche runs the original function again.

## Recursive function

In [7]:
@fleche
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

In [8]:
start = time.time()
fib(20)
end = time.time()
print(f'fib(20) took {end - start:.4f} seconds with caching.')

fib(20) took 0.0066 seconds with caching.


Without caching, this would be much slower as each call to `fib` would be recomputed.

## Passing Digests as Arguments

`fleche` supports passing `Digest` objects directly to cached functions. When a function receives a `Digest`, `fleche` automatically expands it to its actual value from the cache before executing the function. You can use the convenience wrapper `D` to mark a string as a digest.

In [9]:
from fleche import D
from fleche.digest import digest as value_digest

@fleche
def double(x):
    print(f"Doubling {x}...")
    return x * 2

# 1. Run the calculation to ensure it is cached
long_running_calculation(10)

# 2. Compute the value digest for 100 (the cached result)
v = long_running_calculation(10)
val_dig = value_digest(v)
print(f"Value Digest: {val_dig}")

# 3. Pass a short digest prefix of the value to double(); it will expand to 100.
short = str(val_dig)[:8]
print(f"Short digest: {short}")
print(f"Result: {double(D(short))}")

Running calculation for 10...


Value Digest: 60079f7901a9295349d1796c037afc132e81286f785ddeeb763104ef02363102
Short digest: 60079f79
Doubling 100...
Result: 200


## Querying Cached Calls 

### via Function Wrapper

You can retrieve previously cached calls that match some of your function's arguments and metadata using the function wrapper's `query` method. Any field left as `None` is treated as a wildcard. Arguments and result are compared by digest internally, but the wrapper decodes them back to Python objects when returning matches.


Query returns an iterator object, that also defines additional utilities, e.g. to create a table of queried calls, do

In [10]:
long_running_calculation.fleche.query().table()

,name,module,timestart,timestop,walltime
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187
a7ce6824785adc406ca3561dcf98b3c64ddf1539d2467e1b9e6318e4a97368e8,long_running_calculation,__main__,2026-05-20 13:58:11.898600340+00:00,2026-05-20 13:58:13.898777008+00:00,2.000177


The table includes the call digest as an index and the metadata associated with the call.

Arguments can be selectively included in the table via the `arguments`.  You only pay the loading cost for the specified arguments, not for arguments that are not included in the table.

In [11]:
long_running_calculation.fleche.query().table(arguments=['x'])

,name,module,timestart,timestop,walltime,x
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202,2
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187,100
a7ce6824785adc406ca3561dcf98b3c64ddf1539d2467e1b9e6318e4a97368e8,long_running_calculation,__main__,2026-05-20 13:58:11.898600340+00:00,2026-05-20 13:58:13.898777008+00:00,2.000177,10


In [12]:
long_running_calculation.fleche.query().table(arguments=['x'], results=True)

,name,module,result,timestart,timestop,walltime,x
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,4,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202,2
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,10000,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187,100
a7ce6824785adc406ca3561dcf98b3c64ddf1539d2467e1b9e6318e4a97368e8,long_running_calculation,__main__,100,2026-05-20 13:58:11.898600340+00:00,2026-05-20 13:58:13.898777008+00:00,2.000177,10


### via Cache

In [13]:
from fleche.call import QueryCall

In [14]:
cache().query(QueryCall(module="__main__")).table().head()

,name,module,timestart,timestop,walltime
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187
698e29f05ba00ee23503848cd166215f62cd976d54431594036979d8d56f254f,fib,__main__,2026-05-20 13:58:11.887941122+00:00,2026-05-20 13:58:11.887945414+00:00,0.000004
405dfbadf453a9d5bbe3482fbb05251a6fc18b90045f8b4edeafb1c6f236fc80,fib,__main__,2026-05-20 13:58:11.888637781+00:00,2026-05-20 13:58:11.888642073+00:00,0.000004
24231d9bc47f7abc0ea485d178fc8457dce8082790f8c948b936ebe906352225,fib,__main__,2026-05-20 13:58:11.887756109+00:00,2026-05-20 13:58:11.889148235+00:00,0.001392


In [15]:
cache().query(QueryCall(name="fib")).table().head()

,name,module,timestart,timestop,walltime
698e29f05ba00ee23503848cd166215f62cd976d54431594036979d8d56f254f,fib,__main__,2026-05-20 13:58:11.887941122+00:00,2026-05-20 13:58:11.887945414+00:00,0.000004
405dfbadf453a9d5bbe3482fbb05251a6fc18b90045f8b4edeafb1c6f236fc80,fib,__main__,2026-05-20 13:58:11.888637781+00:00,2026-05-20 13:58:11.888642073+00:00,0.000004
24231d9bc47f7abc0ea485d178fc8457dce8082790f8c948b936ebe906352225,fib,__main__,2026-05-20 13:58:11.887756109+00:00,2026-05-20 13:58:11.889148235+00:00,0.001392
dabebddba19859bdb1a075e422b1842dde433c7d6ca51c2b9a284dc1bb743d79,fib,__main__,2026-05-20 13:58:11.887614489+00:00,2026-05-20 13:58:11.889541388+00:00,0.001927
e2b3a4eaf9036f83560677b9bad64cbb8263cf3496ba6faeefc15b4231178912,fib,__main__,2026-05-20 13:58:11.887559414+00:00,2026-05-20 13:58:11.889832497+00:00,0.002273


In [16]:
cache().query(QueryCall(arguments={"x": 100})).table()

,name,module,timestart,timestop,walltime
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187
3fb9bd0ba9c388530e4a0f5fbb789aaf7b5c598c1fae748de84aad8f244a2d8f,double,__main__,2026-05-20 13:58:13.899759531+00:00,2026-05-20 13:58:13.899780273+00:00,0.000021


## Metadata

`fleche` allows you to add metadata to your cached functions using the `tags` context manager. This can be useful for organizing and querying your results.

In [17]:
@fleche
def another_calculation(a, b):
    return a + b

In [18]:
with tags(project='my_project', category='testing'):
    another_calculation(1, 2)
    another_calculation(3, 4)

This metadata is stored alongside the cached result. This metadata can be used to query the cache as well.

In [19]:
# Query by metadata presence (tags) and a specific key-value filter
for call in another_calculation.fleche.query(1, 2, metadata={"tags": {}}):
    # presence-only: any call with 'tags'
    print(call.name, call.arguments, call.metadata.get("tags"))

for call in another_calculation.fleche.query(3, 4, metadata={"tags": {"project": "my_project"}}):
    # equality filter on metadata
    assert call.metadata["tags"]["project"] == "my_project"
    # arguments and result are decoded if they were stored as digests
    print(call.arguments, call.result)


another_calculation LazyArguments({'a': 'da217d50752f3371d9f8b62a3e72409592bd34b74e14fdac43e2b137bd59f21f', 'b': '92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd7798b345471118bf0c67'}) {'project': 'my_project', 'category': 'testing'}
LazyArguments({'a': '65d52a82c5a72f12ca0499522dc9274a0e6822e1038630ba68f94400b3e4c98f', 'b': '83ada2198553b88cb3d0882f7fca8c4e9531049b978df3e9e3b5d6301c6c0bfa'}) 7


## Caching Methods of User-defined Types

`fleche` can also cache methods of classes. For this to work, the class must be "digest-compatible". There are three ways to achieve this:

- Implement a `__digest__` method that returns a `Digest` representing the instance.
- Use a `dataclass` — `fleche` hashes all fields automatically.
- Use an `attrs`-decorated class — the same automatic field hashing applies.

In [20]:
class MyClass:
    def __init__(self, val):
        self.val = val
    
    def __digest__(self):
        # The digest defines how the instance is identified in the cache
        return Digest(str(self.val))

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

In [21]:
obj = MyClass(10)

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

Computing 10 + 5...


Result: 15
First call took 1.00 seconds.
Result: 15
Second call (same instance) took 0.00 seconds.


If you mutate the instance such that its digest changes, the cache will be missed.

In [22]:
obj.val = 20
start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Call after mutation took {time.time() - start:.2f} seconds.")

Computing 20 + 5...


Result: 25
Call after mutation took 1.00 seconds.


### With `dataclass`

`dataclass`-decorated classes are digest-compatible out of the box — no `__digest__` needed.

In [23]:
from dataclasses import dataclass

@dataclass
class MyDataClass:
    val: int

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

dc = MyDataClass(10)

start = time.time()
print(f"Result: {dc.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {dc.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

Computing 10 + 5...


Result: 15
First call took 1.00 seconds.
Result: 15
Second call (same instance) took 0.00 seconds.


### With `attrs`

`attrs`-decorated classes work the same way. Install the optional `attrs` package and `fleche` will hash all `attrs` fields automatically.

In [24]:
import attr

@attr.s
class MyAttrsClass:
    val: int = attr.ib()

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

ac = MyAttrsClass(10)

start = time.time()
print(f"Result: {ac.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {ac.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

Computing 10 + 5...


Result: 15
First call took 1.00 seconds.
Result: 15
Second call (same instance) took 0.00 seconds.


## Passing Digests as Arguments

`fleche` supports passing `Digest` objects directly to cached functions. When a function receives a `Digest`, `fleche` automatically expands it to its actual value from the cache before executing the function. You can use the convenience wrapper `D` to mark a string as a digest.

In [25]:
from fleche import D
from fleche.digest import digest as value_digest

@fleche
def double(x):
    print(f"Doubling {x}...")
    return x * 2

# 1. Run the calculation to ensure it is cached
long_running_calculation(10)

# 2. Compute the value digest for 100 (the cached result)
v = long_running_calculation(10)
val_dig = value_digest(v)
print(f"Value Digest: {val_dig}")

# 3. Pass a short digest prefix of the value to double(); it will expand to 100.
short = str(val_dig)[:8]
print(f"Short digest: {short}")
print(f"Result: {double(D(short))}")

Value Digest: 60079f7901a9295349d1796c037afc132e81286f785ddeeb763104ef02363102
Short digest: 60079f79
Result: 200


## Metadata

`fleche` allows you to add metadata to your cached functions using the `tags` context manager. This can be useful for organizing and querying your results.

In [26]:
@fleche
def another_calculation(a, b):
    return a + b

In [27]:
with tags(project='my_project', category='testing'):
    another_calculation(1, 2)
    another_calculation(3, 4)

This metadata is stored alongside the cached result. You can then use the `metadata.table` method to view the metadata for all cached results.

In [28]:
cache().table()

,name,module,timestart,timestop,walltime,project,category
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202,NaN,NaN
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187,NaN,NaN
698e29f05ba00ee23503848cd166215f62cd976d54431594036979d8d56f254f,fib,__main__,2026-05-20 13:58:11.887941122+00:00,2026-05-20 13:58:11.887945414+00:00,0.000004,NaN,NaN
405dfbadf453a9d5bbe3482fbb05251a6fc18b90045f8b4edeafb1c6f236fc80,fib,__main__,2026-05-20 13:58:11.888637781+00:00,2026-05-20 13:58:11.888642073+00:00,0.000004,NaN,NaN
24231d9bc47f7abc0ea485d178fc8457dce8082790f8c948b936ebe906352225,fib,__main__,2026-05-20 13:58:11.887756109+00:00,2026-05-20 13:58:11.889148235+00:00,0.001392,NaN,NaN
dabebddba19859bdb1a075e422b1842dde433c7d6ca51c2b9a284dc1bb743d79,fib,__main__,2026-05-20 13:58:11.887614489+00:00,2026-05-20 13:58:11.889541388+00:00,0.001927,NaN,NaN
e2b3a4eaf9036f83560677b9bad64cbb8263cf3496ba6faeefc15b4231178912,fib,__main__,2026-05-20 13:58:11.887559414+00:00,2026-05-20 13:58:11.889832497+00:00,0.002273,NaN,NaN
981a54293b3027931a47d21c275955d6dab2fe6d8fa4c2f4fcd3190b6187b0ac,fib,__main__,2026-05-20 13:58:11.887506008+00:00,2026-05-20 13:58:11.890038490+00:00,0.002532,NaN,NaN
fdb0a9b8b1df91293924ad8fc03bcf041f3e50f924e9d199b8606eed60579d91,fib,__main__,2026-05-20 13:58:11.887447834+00:00,2026-05-20 13:58:11.890216112+00:00,0.002768,NaN,NaN
db6b5632c77a01a692e13c486c54536a0c46907972d93900a60a1eab59110b93,fib,__main__,2026-05-20 13:58:11.887393475+00:00,2026-05-20 13:58:11.890395880+00:00,0.003002,NaN,NaN


## Filtering

The metadata table is just pandas so you can query and filter as you like.

In [29]:
cache().table().query('name!="fib"')

,name,module,timestart,timestop,walltime,project,category
4435e2927c194eff1bb90270ab3f353f0db1a8b50ec04932de88f7a4e5745953,long_running_calculation,__main__,2026-05-20 13:58:07.866472244+00:00,2026-05-20 13:58:09.866674185+00:00,2.000202,NaN,NaN
cb9db168b7a8eb99bda637a54241ad5d4e2726b5cd67ad4cfb21baa5874c2b9e,long_running_calculation,__main__,2026-05-20 13:58:09.876649618+00:00,2026-05-20 13:58:11.876836300+00:00,2.000187,NaN,NaN
a7ce6824785adc406ca3561dcf98b3c64ddf1539d2467e1b9e6318e4a97368e8,long_running_calculation,__main__,2026-05-20 13:58:11.898600340+00:00,2026-05-20 13:58:13.898777008+00:00,2.000177,NaN,NaN
3fb9bd0ba9c388530e4a0f5fbb789aaf7b5c598c1fae748de84aad8f244a2d8f,double,__main__,2026-05-20 13:58:13.899759531+00:00,2026-05-20 13:58:13.899780273+00:00,0.000021,NaN,NaN
a8a3061653183cff08e5c414f9a4087550ab5e33cc186ae7a2e91aa1fbac8c80,another_calculation,__main__,2026-05-20 13:58:13.981916666+00:00,2026-05-20 13:58:13.981927633+00:00,0.000011,my_project,testing
1e352b538b9219d4e5fcaf429882a4b3587f3bc6358ef0feca453b301b40156a,another_calculation,__main__,2026-05-20 13:58:13.982227802+00:00,2026-05-20 13:58:13.982236147+00:00,0.000008,my_project,testing
ae154f37c2176451bd04c82e702bd934f1ecfdb5d49698d94ba6be4b64e1be86,MyClass.compute,__main__,2026-05-20 13:58:13.998702288+00:00,2026-05-20 13:58:14.998900652+00:00,1.000198,NaN,NaN
34818cf0647e22b9f7e0d46adac2fd394b76e572f132976946f2c76498a6140f,MyClass.compute,__main__,2026-05-20 13:58:15.004741192+00:00,2026-05-20 13:58:16.004941463+00:00,1.000200,NaN,NaN
5fef928ab57118f9390f0627ba47fe2030d9287e5b05826292042b5a41a4ec06,MyDataClass.compute,__main__,2026-05-20 13:58:16.012222767+00:00,2026-05-20 13:58:17.012420893+00:00,1.000198,NaN,NaN
f240e0fe816bfa4476a5ef4e5de2c98d4b5276da3fb10e0359b0c3903ae8c466,MyAttrsClass.compute,__main__,2026-05-20 13:58:17.020026207+00:00,2026-05-20 13:58:18.020214558+00:00,1.000188,NaN,NaN


## Querying Cached Calls via Function Wrapper

You can retrieve previously cached calls that match some of your function's arguments and metadata using the function wrapper's `query` method. Any field left as `None` is treated as a wildcard. Arguments and result are compared by digest internally, but the wrapper decodes them back to Python objects when returning matches.

Example using the `another_calculation` wrapper we created above:


In [30]:
# Query by metadata presence (tags) and a specific key-value filter
for call in another_calculation.fleche.query(1, 2, metadata={"tags": {}}):
    # presence-only: any call with 'tags'
    print(call.name, call.arguments, call.metadata.get("tags"))

for call in another_calculation.fleche.query(3, 4, metadata={"tags": {"project": "my_project"}}):
    # equality filter on metadata
    assert call.metadata["tags"]["project"] == "my_project"
    # arguments and result are decoded if they were stored as digests
    print(call.arguments, call.result)


another_calculation LazyArguments({'a': 'da217d50752f3371d9f8b62a3e72409592bd34b74e14fdac43e2b137bd59f21f', 'b': '92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd7798b345471118bf0c67'}) {'project': 'my_project', 'category': 'testing'}
LazyArguments({'a': '65d52a82c5a72f12ca0499522dc9274a0e6822e1038630ba68f94400b3e4c98f', 'b': '83ada2198553b88cb3d0882f7fca8c4e9531049b978df3e9e3b5d6301c6c0bfa'}) 7


## Lazy Loading

When you load a call from the cache, fleche returns it as a `LazyCall` by default. Arguments and results are only fetched from storage when you actually access them — so iterating over a large cache or inspecting metadata stays fast even when individual results are huge.


In [31]:
# Default load: returns a LazyCall — no deserialization yet
key = fib.fleche.digest(20)
lazy_call = cache().load(key)
print(f"Got a {type(lazy_call).__name__} for {lazy_call.name}(20)")

# Accessing .result triggers the actual load from storage
print(f"Result: {lazy_call.result}")


Got a LazyCall for fib(20)
Result: 6765


To load everything upfront, call `.fetch()` on a lazy call you already have.


In [32]:
# Fetch everything upfront from a lazy call:
lazy_call.fetch()


Call(name='fib', arguments={'n': 20}, metadata=defaultdict(<class 'dict'>, {'runtime': {'timestart': 1779285491.8865771, 'timestop': 1779285491.8929136, 'walltime': 0.0063364505767822266}}), module='__main__', version=None, code_digest=None, result=6765)